**TOP 1000 FEATURES SHAP**

In [0]:
import pandas as pd

In [0]:
df = spark.table(
    "`ph-com-ai-us-prod-catalog-2553213839599380`."
    "kerendia_oe_journey."
    "final_mdb_oe_2_0_21_instances"
)

print("Rows:", mdb.count())
print("Columns:", len(mdb.columns))

In [0]:
df.limit(5).display()

In [0]:
# Case-insensitive column search

search_text = "TOTAL_NBRX_C4".lower()

matching_cols = [
    c for c in df.columns
    if search_text in c.lower()
]

print("Total columns:", len(df.columns))
print("Matching columns:", len(matching_cols))

for c in matching_cols:
    print(c)

In [0]:
!pip install openpyxl
shap_final= pd.read_excel("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/Top Features OE 2.0.xlsx")


In [0]:
shap_final.head(5)

In [0]:
# shap_final[shap_final['shap_rank']==500]

In [0]:
# shap_final[shap_final['shap_rank']==1000]

In [0]:
DATA_PATH = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_final_parquet"
)

mdb = spark.read.parquet(DATA_PATH)

# Check shape
print("Rows:", mdb.count())
print("Columns:", len(mdb.columns))

# Show top rows
mdb.limit(5).display()

In [0]:
col_name = "INSTANCE"

if col_name in mdb.columns:
    print(f"{col_name} exists in mdb")
else:
    print(f"{col_name} does not exist in mdb")

In [0]:
from pyspark.sql import functions as F

# Extra columns to always keep
extra_cols = ["TOTAL_NBRX_C4", "PROVIDER_ID", "INSTANCE"]

# Get feature names from pandas shap_final
feature_cols = (
    shap_final["feature"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Keep only columns that exist in mdb
mdb_cols = set(mdb.columns)

final_cols = [
    c for c in feature_cols + extra_cols
    if c in mdb_cols
]

# Remove duplicates while preserving order
final_cols = list(dict.fromkeys(final_cols))

# Filter mdb and store back in mdb
mdb = mdb.select(
    *[
        F.col("`" + c.replace("`", "``") + "`").alias(c)
        for c in final_cols
    ]
)

# Check shape
print("Rows:", mdb.count())
print("Columns:", len(mdb.columns))

In [0]:
mdb.limit(5).display()

In [0]:
col_name = "TOTAL_NBRX_C4"

if col_name in mdb.columns:
    print(f"{col_name} exists in mdb")
else:
    print(f"{col_name} does not exist in mdb")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType
)

target_col = "TOTAL_NBRX_C4"
provider_col = "PROVIDER_ID"

def qcol(c):
    return F.col("`" + c.replace("`", "``") + "`")

def clean_double(c):
    x = qcol(c).cast("double")
    return F.when(x.isNull() | F.isnan(x), None).otherwise(x)

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

# ============================================================
# Settings
# ============================================================

chunk_size = 50
percentile_accuracy = 10000

total_rows = mdb.count()

y = clean_double(target_col)

result_rows = []

# ============================================================
# Calculate summaries per feature
# ============================================================

for feature_chunk in chunks(feature_cols, chunk_size):

    agg_exprs = []

    for i, feature in enumerate(feature_chunk):

        x = clean_double(feature)
        alias = f"f_{i}"

        agg_exprs.extend([
            F.mean(x).alias(f"{alias}__mean"),
            F.percentile_approx(x, 0.50, percentile_accuracy).alias(f"{alias}__median"),
            F.countDistinct(x).alias(f"{alias}__nunique"),
            F.countDistinct(F.when(x.isNotNull(), qcol(provider_col))).alias(f"{alias}__nunique_provider_id"),
            F.percentile_approx(x, 0.25, percentile_accuracy).alias(f"{alias}__percentile25th"),
            F.percentile_approx(x, 0.75, percentile_accuracy).alias(f"{alias}__percentile75th"),
            F.stddev_samp(x).alias(f"{alias}__standard_deviation"),
            F.sum(F.when(x.isNotNull(), 1).otherwise(0)).alias(f"{alias}__non_null_rows"),
            F.sum(F.when(x == 0, 1).otherwise(0)).alias(f"{alias}__zero_value_count"),
            F.sum(F.when(x.isNotNull() & (x != 0), 1).otherwise(0)).alias(f"{alias}__non_zero_value_count"),
            F.corr(x, y).alias(f"{alias}__corr_with_y")
        ])

    row_dict = mdb.agg(*agg_exprs).collect()[0].asDict()

    for i, feature in enumerate(feature_chunk):

        alias = f"f_{i}"

        non_null_rows = row_dict[f"{alias}__non_null_rows"]

        fill_rate_pct = (
            non_null_rows / total_rows * 100
            if total_rows > 0 and non_null_rows is not None
            else None
        )

        result_rows.append((
            feature,
            row_dict[f"{alias}__mean"],
            row_dict[f"{alias}__median"],
            row_dict[f"{alias}__nunique"],
            row_dict[f"{alias}__nunique_provider_id"],
            row_dict[f"{alias}__percentile25th"],
            row_dict[f"{alias}__percentile75th"],
            total_rows,
            non_null_rows,
            fill_rate_pct,
            row_dict[f"{alias}__standard_deviation"],
            row_dict[f"{alias}__zero_value_count"],
            row_dict[f"{alias}__non_zero_value_count"],
            row_dict[f"{alias}__corr_with_y"]
        ))

# ============================================================
# Create final Spark dataframe
# ============================================================

schema = StructType([
    StructField("feature", StringType(), True),
    StructField("mean", DoubleType(), True),
    StructField("median", DoubleType(), True),
    StructField("nunique", LongType(), True),
    StructField("nunique_PROVIDER_ID", LongType(), True),
    StructField("percentile25th", DoubleType(), True),
    StructField("percentile75th", DoubleType(), True),
    StructField("n_rows", LongType(), True),
    StructField("non_null_rows", LongType(), True),
    StructField("fill_rate_pct", DoubleType(), True),
    StructField("standard_deviation", DoubleType(), True),
    StructField("zero_value_count", LongType(), True),
    StructField("non_zero_value_count", LongType(), True),
    StructField("corr_with_TOTAL_NBRX_C4", DoubleType(), True)
])

feature_summary_df = spark.createDataFrame(result_rows, schema)


In [0]:
feature_summary_df.limit(5).display()

In [0]:
# ============================================================
# Save as single named CSV file
# ============================================================

tmp_path = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "_tmp_feature_summary_new_25k_csv"
)

final_csv_path = (
    "dbfs:/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_summary_new_25k.csv"
)

# Write temporary single-part CSV
feature_summary_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(tmp_path)

# Find the generated part file
part_file = [
    f.path for f in dbutils.fs.ls(tmp_path)
    if f.name.startswith("part-") and f.name.endswith(".csv")
][0]

# Copy part file to final CSV name
dbutils.fs.cp(part_file, final_csv_path, True)

# Remove temp folder
dbutils.fs.rm(tmp_path, True)

print("Single CSV saved at:")
print(final_csv_path)

In [0]:
import pandas as pd

summary_path = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_summary_new_25k.csv"
)

summary = pd.read_csv(summary_path)

print(summary.shape)
summary.head()

In [0]:
summary.to_csv("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/shap_1k_new2_eda.csv",index=False)